In [5]:
import pandas as pd
import numpy as np

In [6]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [7]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [8]:
X = df.drop(columns="Class")
y = df["Class"]

In [12]:
# Encoding y labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

y = le.fit_transform(y)

In [14]:
# Train, Test & Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [15]:
# Scaling the Data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [19]:
# Here we start deep learning
# Make Tensors
import torch
import torch.nn as nn

X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long)

X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [20]:
# TensorDataSet & DataLoader
from torch.utils.data import TensorDataset, DataLoader
# tensorDataset
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)
# DataLoader
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [23]:
# Define ANN
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            # 1st hidden layer
            nn.Linear(X.shape[1],64),
            nn.ReLU(),

            # 2nd hidden layer
            nn.Linear(64,64),
            nn.ReLU(),

            # Output layer
            nn.Linear(64,7),
        )
    
    def forward(self,x):
        return self.model(x)

In [24]:
model = ANN()
# Loss & Optimizer
import torch.optim as optim
# loss funx
criteria = nn.CrossEntropyLoss()
# Optimizer
optimizer = optim.Adam(model.parameters())

In [25]:
# Training Model
epochs = 100

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for xb,yb in train_loader:
        optimizer.zero_grad()
        output = model(xb)
        loss = criteria(output,yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss / len(train_loader)
    print(f"epoch {epoch+1}/{epochs}, loss : {train_loss}")    

epoch 1/100, loss : 1.6545517703761226
epoch 2/100, loss : 1.0914011286652607
epoch 3/100, loss : 0.7201334263967432
epoch 4/100, loss : 0.5468502394531084
epoch 5/100, loss : 0.44563283220581384
epoch 6/100, loss : 0.3957897100759589
epoch 7/100, loss : 0.3453642551017844
epoch 8/100, loss : 0.3069939082083495
epoch 9/100, loss : 0.28493211224027304
epoch 10/100, loss : 0.2562225687762965
epoch 11/100, loss : 0.24328777323598447
epoch 12/100, loss : 0.22499345372552457
epoch 13/100, loss : 0.21632299345472586
epoch 14/100, loss : 0.20311715972164404
epoch 15/100, loss : 0.19014142810002618
epoch 16/100, loss : 0.18382897584334665
epoch 17/100, loss : 0.18544226982023404
epoch 18/100, loss : 0.18474163341781366
epoch 19/100, loss : 0.1648941642564276
epoch 20/100, loss : 0.15555541379296262
epoch 21/100, loss : 0.1533365667514179
epoch 22/100, loss : 0.16128215128960816
epoch 23/100, loss : 0.1534573874719765
epoch 24/100, loss : 0.13875054503264633
epoch 25/100, loss : 0.1322423976076

In [26]:
# Evaluate
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for xb,yb in test_loader:
        output = model(xb)
        _,predicted = torch.max(output,1)

        correct += (predicted == yb).sum().item()
        total += yb.size(0)

print(f"Total : {total}")
print(f"Correct Predicted : {correct}")
print(f"Accuracy : {correct/total}")

Total : 180
Correct Predicted : 171
Accuracy : 0.95
